# Plots

In [ ]:
library(readr)
library(purrr)
library(dplyr)
library(stringr)
library(fuzzyjoin)
library(ggplot2)
library(dplyr)
library(tidyverse)
library(Matrix)
library(reshape2)
library(RColorBrewer)

In [ ]:
options(repr.matrix.max.cols = Inf,  # show all columns
        repr.matrix.max.rows = 200)  # adjust rows as you like

## 0. Plot parameters

In [ ]:
out_dir = "/ceph.groups/mshahbazi.grp/rsakata/EXP58/output/setA_T3/4_plots"
root_dir = "/ceph.groups/mshahbazi.grp/rsakata/EXP58/output/setA_T3/1_cellpose"
sample_sheet_csv = "/ceph.groups/mshahbazi.grp/rsakata/EXP58/sample_sheet.csv"
cyto_csv ="/ceph.groups/mshahbazi.grp/rsakata/EXP58/output/setA_T3/3_cytomask_intensities/setA_cyto_intensity.csv"

filter_dapi = FALSE

In [ ]:
EXP = "EXP58"
EXP_SUB = "setA_T3"

In [ ]:
# define const for visualization
FONT.SIZE <- 9
LABEL.FONT.SIZE <- 9
w <- 2
h <- 2.5
LINE.W <- 0.3 # equivalent to 0.75pt in Keynote

settheme <- theme_minimal() + theme(
    panel.background = element_blank(),
    panel.grid.major = element_blank(), 
    panel.grid.minor = element_blank(),
    plot.background = element_blank(),
    axis.ticks = element_line(colour = "black", linewidth = LINE.W),
    axis.line = element_line(linewidth = LINE.W, colour = "black"),
    #axis.line.y = element_blank(),
    axis.title = element_text(size = FONT.SIZE),
    axis.text = element_text(colour = "black", size = FONT.SIZE),
    axis.text.x = element_text(colour = "black", angle = 0,size = LABEL.FONT.SIZE),
    legend.position="right",
    title = element_text(size = FONT.SIZE))

In [ ]:
col_aneu = c("euploid"= "#D4D1B3","monosomy"="#109E9D","trisomy"="#F26B3B","complex"= "#886DB0")

col_condition_2 = c("control" = "#285F62", 
               "reversine" = "#CA4F33", 
               "mosaic"= "#E2A557")

col_condition = c("G_R"= "#5E5E5E","Grev_R"="#86AB30","Rrev_G"="#EB5951", "Grev_Rrev"="#F0A329")

col_GFP = c("TRUE"= "#86AB30","FALSE"="#8d8d8dff")


## 1. Extract summary files

In [ ]:
if (!dir.exists(out_dir)) { 
  dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)
 }

In [ ]:
# Find matching CSVs in all subfolders
csv_paths <- list.files(
  path = root_dir,
  pattern = "\\.csv$",
  recursive = TRUE,
  full.names = TRUE
)

if (length(csv_paths) == 0) stop("No matching files found.")

# Read and bind
combined_df <- purrr::map_dfr(csv_paths, ~ readr::read_csv(.x, show_col_types = FALSE))


#rename colums and add new column for sample number
combined_df <- combined_df %>%
  rename(image = sample) %>%
  mutate(sample = str_replace(image, "_.*$", ""))

combined_df$sample <- as.character(combined_df$sample)


In [ ]:
head(combined_df)

In [ ]:
#merge with sample sheet
sample_sheet <- read_csv(sample_sheet_csv, show_col_types = FALSE)
sample_sheet$sample <- as.character(sample_sheet$sample)

merged_df <- combined_df %>%
  left_join(sample_sheet, by = "sample")   # keeps all rows from df1

In [ ]:
sample_sheet

In [ ]:
head(merged_df)

In [ ]:
#merge with cyto intensity
cyto_df <- read_csv(cyto_csv, show_col_types = FALSE)
sample_sheet$sample <- as.character(sample_sheet$sample)

# join in sample in sample sheet is contained in combined df sample
merged_df <- regex_left_join(
  merged_df,
  cyto_df,
  by = c("file_name" = "file"),
  ignore_case = FALSE
) 

In [ ]:
tbl <- merged_df %>%
  group_by(sample, sample_name) %>%
  summarise(n_images = n_distinct(image), .groups = "drop") %>%
  arrange(sample)  # optional

tbl

In [ ]:
merged_df$exp = EXP
merged_df$exp_sub = EXP_SUB

## 2. Preprocess

In [ ]:
colnames(merged_df)

### A) background_intensity

In [ ]:
merged_long <- merged_df %>%
  pivot_longer(
    cols = c(dapi_mean, GATA3_mean, GFP_mean, NANOG_mean),
    names_to = "channel",
    values_to = "mean_value"
  )

In [ ]:
title = "A_overview_background_intensity"
w <- 3.5
h <- 5
options(repr.plot.width=w, repr.plot.height=h)
  
p = ggplot(merged_long, aes(x = sample_name, y =mean_value)) +  # dots for each file
    stat_summary(
      fun = mean, 
      geom = "bar",
      position = position_dodge(width = 0.75),
      fill = "grey", alpha = 0.4, width = 0.6) +   # error bars
    geom_jitter(
      aes(fill = sample_name),
      position = position_jitterdodge(jitter.width = 0.1, dodge.width = 0),
      size = 0.5, alpha = 0.8
    )  +
    labs(
      title = title,
      y = "intensity",
      x = ""
    )+ settheme+
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1)
    ) +
     # scale_y_continuous(limits = c(0, 700), expand = c(0, 0))+ 
      facet_wrap(~channel, ncol = 1)
      #scale_color_manual(values=col_sample_name)

ggsave(file.path(out_dir, sprintf("%s.pdf", title)),
                plot = p, width = w, height = h)

p

### B) background intensity per sample

In [ ]:
title = "B1_GATA3_background_intensity"
w <- 6
h <- 6
options(repr.plot.width=w, repr.plot.height=h)

p = ggplot(cyto_df, aes(x = GATA3_mean,
               y = file)) +
  geom_col() +
  labs(x = "intensity", y = "", title = "GATA3 intensity") +
  settheme

ggsave(file.path(out_dir, sprintf("%s.pdf", title)),
                plot = p, width = w, height = h)

p

In [ ]:
## Fill NA for GATA3 if cytointensity greater than 100
merged_df <- merged_df %>%
  mutate(Mean_GATA3 = replace(Mean_GATA3, GATA3_mean > 100, NA_real_))


### Normalise intensities

In [ ]:
title = "gata3_dapi"

w <- 3.5
h <- 2.5
options(repr.plot.width=w, repr.plot.height=h)

ggscatter = ggplot(merged_df, aes(x = Mean_dapi, y = Mean_GATA3, color = condition)) +
  geom_point(alpha = 0.6, size = 0.6) +
  labs(x = " Mean_dapi", y = " Mean_GATA3", title = "") +
  settheme+
  scale_color_manual(values=col_condition)
ggsave(plot = ggscatter, filename = sprintf("%s/%s.pdf",out_dir, title), w = w, h = h)
ggscatter

In [ ]:
title = "nanog_dapi"

w <- 3.5
h <- 2.5
options(repr.plot.width=w, repr.plot.height=h)

ggscatter = ggplot(merged_df, aes(x = Mean_dapi, y = Mean_NANOG, color = condition)) +
  geom_point(alpha = 0.6, size = 0.6) +
  labs(x = " Mean_dapi", y = " Mean_NANOG", title = "") +
  settheme+
  scale_color_manual(values=col_condition)
ggsave(plot = ggscatter, filename = sprintf("%s/%s.pdf",out_dir, title), w = w, h = h)
ggscatter

In [ ]:
#Compute the normalised intensities
merged_df$GATA3_norm = (merged_df$Mean_GATA3-merged_df$GATA3_mean)/merged_df$Mean_dapi
merged_df$GFP_norm = merged_df$Mean_GFP/merged_df$Mean_dapi
merged_df$NANOG_norm = (merged_df$Mean_NANOG-merged_df$NANOG_mean)/merged_df$Mean_dapi

## Plot 

### C) Intensity threshold jitter

In [ ]:
library(ggplot2)
library(rlang)

plot_jitter <- function(
  data,
  x = sample_name,
  y = GATA3_norm,
  color = condition,
  out_dir,
  title = "",
  w = 5, h = 3,
  palette = NULL,
  hline_at = NULL,                 # <— add a dotted horizontal line at this y
  hline_color = "gray30",
  hline_size = 0.5,
  hline_lty =  "dashed"
) {
  x <- rlang::enquo(x); y <- rlang::enquo(y); color <- rlang::enquo(color)

  p <- ggplot(data, aes(x = fct_rev(!!x), y = !!y, color = !!color)) +
    geom_jitter(width = 0.2, size = 0.7, alpha = 0.6, na.rm = TRUE) +
    labs(x = "", y = "normalised intensity", title = title) +
    settheme +
    scale_y_continuous(limits = c(0, NA), expand = c(0, 0)) +
    coord_flip()

  if (!is.null(palette)) p <- p + scale_color_manual(values = palette)
  if (!is.null(hline_at)) p <- p + geom_hline(yintercept = hline_at, linetype = hline_lty,
                                              linewidth = hline_size, color = hline_color)

  ggsave(file.path(out_dir, sprintf("%s.pdf", title)), plot = p, width = w, height = h)
  p
}


### D) Intensity threshold_hist

In [ ]:
plot_hist <- function(
  data,
  x = GATA3_norm,
  facet = sample_name,
  out_dir,
  title = "GATA3_norm_hist",
  w = 5, h = 5,
  bins = 30,
  binwidth = NULL,
  fill = "#6CD1D4",
  outline = "gray10",
  linewidth = 0.2,
  free_y = TRUE,
  vline_at = NULL,              # <— vertical line at this x
  vline_color = "gray30",
  vline_size = 0.5,
  vline_lty = "dashed"
) {
  x     <- rlang::enquo(x)
  facet <- rlang::enquo(facet)

  p <- ggplot2::ggplot(data, ggplot2::aes(x = !!x)) +
    (if (!is.null(binwidth))
       ggplot2::geom_histogram(binwidth = binwidth, boundary = 0, closed = "left",
                                na.rm = TRUE, fill = fill, color = outline, linewidth = linewidth)
     else
       ggplot2::geom_histogram(bins = bins, na.rm = TRUE,
                                fill = fill, color = outline, linewidth = linewidth)) +
    ggplot2::labs(x = "normalised intensity", y = "Count", title = title) +
    settheme +
    ggplot2::scale_y_continuous(limits = c(0, NA), expand = c(0, 0),
      breaks = scales::pretty_breaks(n = 2)   # <— fewer ticks
    )  +
    ggplot2::facet_grid(rows = ggplot2::vars(!!facet),
                        scales = if (free_y) "free_y" else "fixed",
      switch = "y"            ) +
    ggplot2::theme(
      strip.text.y.left = ggplot2::element_text(angle = 0, hjust = 0),
      strip.background = ggplot2::element_blank(),
      strip.placement  = "outside",
      plot.margin = ggplot2::margin(5.5, 5.5, 5.5, 35, "pt")
    )

  if (!is.null(vline_at)) {
    p <- p + ggplot2::geom_vline(xintercept = vline_at,
                                 linetype = vline_lty,
                                 linewidth = vline_size,
                                 color = vline_color)
  }

  ggplot2::ggsave(file.path(out_dir, sprintf("%s.pdf", title)),
                  plot = p, width = w, height = h)
  p
}




In [ ]:
#dapi
w <- 4
h <- 3
options(repr.plot.width=w, repr.plot.height=h)

DAPI_thresh = 150

gghist <- plot_hist(
  data    = merged_df,
  x       = Mean_dapi,
  facet   = sample_name,
  out_dir = out_dir,
  binwidth = 0.1,
  title   = "D_Mean_dapi_hist",
  fill = "#362fffff",
  w = w, h = h,
  vline_at = DAPI_thresh
)
gghist

In [ ]:
# Filter cells with high dapi levels

merged_df <- merged_df %>%
  mutate(
    DAPIpos = Mean_dapi < DAPI_thresh
  )

if (filter_dapi) {
  merged_df <- merged_df %>% 
    filter(Mean_dapi < DAPI_thresh)
}

In [ ]:
NANOG_thresh = 0.3

In [ ]:
#NANOG

w <- 5
h <- 2.5
options(repr.plot.width=w, repr.plot.height=h)
ggjitter <- plot_jitter(
  data = merged_df,
  x = sample_name,
  y = NANOG_norm,
  color = condition,
  out_dir = out_dir,
  title = "C_NANOG_norm_intensity",
  w = 5, h = 3,
  palette = col_condition,
  hline_at = NANOG_thresh
)
ggjitter

gghist <- plot_hist(
  data    = merged_df,
  x       = NANOG_norm,
  facet   = sample_name,
  out_dir = out_dir,
  binwidth = 0.1,
  title   = "D_NANOG_norm_hist",
  fill = "#FF2F92",
  w = 5, h = 3,
  vline_at = NANOG_thresh
)
gghist


In [ ]:
w <- 5
h <- 3
options(repr.plot.width=w, repr.plot.height=h)

GATA3_thresh = 0.4

gg_jitter <- plot_jitter(
  data = merged_df,
  x = sample_name,
  y = GATA3_norm,
  color = condition,
  out_dir = out_dir,
  title = "C_GATA3_norm_intensity",
  w = 5, h = 3,
  palette = col_condition,
  hline_at = GATA3_thresh
)
gg_jitter

gg_hist <- plot_hist(
  data    = merged_df,
  x       = GATA3_norm,
  facet   = sample_name,
  out_dir = out_dir,
  binwidth = 0.1,
  title   = "D_GATA3_norm_hist",
  fill = "#6CD1D4",
  w = 5, h = 3,
  vline_at = GATA3_thresh)
gg_hist

In [ ]:
title = "gfp_dapi"

w <- 5
h <- 4
options(repr.plot.width=w, repr.plot.height=h)
GFP_thresh = 2

ggscatter = ggplot(merged_df, aes(x = Mean_dapi, y =  Mean_GFP, color = condition)) +
  geom_point(alpha = 0.6, size = 0.1) +
  labs(x = " Mean_dapi", y = " Mean_gfp", title = "") +
  settheme+
  scale_color_manual(values=col_condition)+ 
  facet_wrap(~sample)+
  geom_abline(slope = GFP_thresh, intercept = 0, linetype = "dashed", color = "grey20") 
ggsave(plot = ggscatter, filename = sprintf("%s/%s.pdf",out_dir, title), w = w, h = h)
ggscatter

In [ ]:
w <- 5
h <- 3
options(repr.plot.width=w, repr.plot.height=h)

gg_jitter <- plot_jitter(
  data = merged_df,
  x = sample_name,
  y = GFP_norm,
  color = condition,
  out_dir = out_dir,
  title = "C_GFP_norm_intensity",
  w = 5, h = 3,
  palette = col_condition,
  hline_at = GFP_thresh
)
gg_jitter

gg_hist <- plot_hist(
  data    = merged_df,
  x       = GFP_norm,
  facet   = sample_name,
  out_dir = out_dir,
  binwidth = 0.1,
  title   = "D_GFP_norm_hist",
  fill = "#6EA537",
  w = 5, h = 3,
  vline_at = GFP_thresh
)
gg_hist

In [ ]:
title = "GATA3_NANOG_scatter"

w <- 6
h <- 6
options(repr.plot.width=w, repr.plot.height=h)

ggscatter = ggplot(merged_df, aes(x = NANOG_norm, y = GATA3_norm, color = condition)) +
  geom_point(alpha = 0.6, size = 0.6) +
  labs(x = " NANOG_norm", y = " GATA3_norm", title = title) +
  settheme+
   scale_color_manual(values=col_condition)+
  #geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "blue") 
  facet_wrap(~sample_name)

ggsave(plot = ggscatter, filename = sprintf("%s/%s.pdf",out_dir, title), w = w, h = h)
ggscatter

### E) %pos marker

In [ ]:
plot_pct_bar_points <- function(
  data,                               # e.g., summary_df
  pct = pct_GATA3,                    # <-- column with % values to plot
  sample = sample_name,               # sample/category column
  condition = condition,              # grouping/fill column
  out_dir,                    # folder to save (optional)
  title = NULL,                       # default built from pct col name if NULL
  palette = NULL,                     # named vector for fill
  w = 4, h = 2,
  y_max = 110,
  y_ticks = 5,
  bar_width = 0.6,
  point_size = 1.1,
  point_alpha = 0.7,
  jitter_width = 0.05
) {
  pct      <- enquo(pct)
  sample   <- enquo(sample)
  condition<- enquo(condition)

  # default title from pct column name if not supplied
  if (is.null(title)) {
    title <- paste0(as_label(pct), "+")
  }

  # per-sample means (by condition) for the chosen pct column
  means_df <- data %>%
    group_by(!!sample, !!condition) %>%
    summarise(mean_pct = mean(!!pct, na.rm = TRUE), .groups = "drop")

  # build plot (reverse sample order, flip coords)
  p <- ggplot(data, aes(x = fct_rev(!!sample), y = !!pct)) +
    geom_col(
      data = means_df,
      aes(y = mean_pct, fill = !!condition),
      width = bar_width
    ) +
    geom_point(
      size = point_size, alpha = point_alpha,
      position = position_jitter(width = jitter_width),
      na.rm = TRUE
    ) +
    labs(x = "", y = "% positive", title = title, fill = rlang::as_name(condition)) +
    settheme +
    scale_y_continuous(limits = c(0, y_max), expand = c(0, 0),
                       breaks = scales::pretty_breaks(y_ticks)) +
    coord_flip()

  if (!is.null(palette)) {
    p <- p + scale_fill_manual(values = palette)
  }

  safe_title <- gsub("[^[:alnum:]_\\-]+","_", title)
  ggplot2::ggsave(file.path(out_dir, sprintf("%s.pdf", safe_title)),
                  plot = p, width = w, height = h)
  options(repr.plot.width=w, repr.plot.height=h)
  p
}

In [ ]:
# thresholds (edit if you want different cutoffs)
thr <- list(
  GATA3_norm = GATA3_thresh,
  NANOG_norm = NANOG_thresh,
  GFP_norm = GFP_thresh
)

summary_df <- merged_df %>%
  group_by(image, sample_name, condition) %>%
  summarise(
    n = n(),
    pct_GATA3 = 100 * mean(GATA3_norm > thr$GATA3_norm, na.rm = TRUE),
    pct_NANOG = 100 * mean(NANOG_norm > thr$NANOG_norm, na.rm = TRUE),
    pct_GFP = 100 * mean(GFP_norm > thr$GFP_norm, na.rm = TRUE),
    pct_negative = 100 * mean(!(NANOG_norm > thr$NANOG_norm | GATA3_norm > thr$GATA3_norm)),
    pct_double_GATA3_NANOG = 100 * mean(GATA3_norm > thr$GATA3_norm & NANOG_norm > thr$NANOG_norm),
    .groups = "drop"
  )

head(summary_df)


In [ ]:
# Plot % GATA3+
plot_pct_bar_points(summary_df, pct = pct_GATA3, palette = col_condition, out_dir = out_dir, 
                    title = "E_pctGATA3+")

# Plot % NANOG+
plot_pct_bar_points(summary_df, pct = pct_NANOG, palette = col_condition,out_dir = out_dir,
                    title = "E_pctNANOG+")
# Plot % GFP+
plot_pct_bar_points(summary_df, pct = pct_GFP, palette = col_condition,out_dir = out_dir,
                    title = "E_pctGFP+")

# Plot % neg
plot_pct_bar_points(summary_df, pct = pct_negative, palette = col_condition,out_dir = out_dir, 
                    title = "E_pctnegative", y_max = 40)

# Plot % double+
plot_pct_bar_points(summary_df, pct = pct_double_GATA3_NANOG, palette = col_condition,out_dir = out_dir, 
                    title = "E_pct_GATA3+_NANOG+", y_max = 40)



### F) Intensity per subset

In [ ]:
# find average intensities after subtyping
merged_df <- merged_df %>%
  mutate(
    GATA3pos = GATA3_norm > GATA3_thresh,
    NANOGpos = NANOG_norm > NANOG_thresh,
    GFPpos = GFP_norm > GFP_thresh
  )


In [ ]:
title   <- "F_GATA3_norm_intensity_in GATA3pos"
w <- 5; h <- 3
options(repr.plot.width=w, repr.plot.height=h)
palette <- col_condition

p <- ggplot(merged_df %>% filter(GATA3pos == TRUE), aes(x = forcats::fct_rev(sample_name), y = GATA3_norm, color = condition)) +
  # boxplots per condition (behind points)
  geom_boxplot(
    aes(fill = condition),
    width = 0.5, alpha = 0.25, outlier.shape = NA,
    position = position_dodge2(width = 0.6, preserve = "single")
  ) +
  # points with jitter+dodge (no width/height here)
  geom_point(
    size = 0.7, alpha = 0.6,
    position = position_jitterdodge(jitter.width = 0.2, dodge.width = 0.6),
    na.rm = TRUE
  ) +
  labs(x = "", y = "GATA3 intensity (normalized)", title = title) +
  settheme +
  #scale_y_continuous(limits = c(0, NA), expand = c(0, 0)) +
  coord_flip() +
  scale_color_manual(values = palette) +
  scale_fill_manual(values = palette, guide = "none")  # avoid duplicate legend

ggsave(file.path(out_dir, sprintf("%s.pdf", title)), plot = p, width = w, height = h)
p


In [ ]:
title   <- "F_NANOG_norm_intensity_in_NANOGpos"
w <- 5; h <- 3
options(repr.plot.width=w, repr.plot.height=h)
palette <- col_condition

p <- ggplot(gata3_neg <- merged_df %>% filter(NANOGpos == TRUE), aes(x = forcats::fct_rev(sample_name), y = NANOG_norm, color = condition)) +
  # boxplots per condition (behind points)
  geom_boxplot(
    aes(fill = condition),
    width = 0.5, alpha = 0.25, outlier.shape = NA,
    position = position_dodge2(width = 0.6, preserve = "single")
  ) +
  # points with jitter+dodge (no width/height here)
  geom_point(
    size = 0.7, alpha = 0.6,
    position = position_jitterdodge(jitter.width = 0.2, dodge.width = 0.6),
    na.rm = TRUE
  ) +
  labs(x = "", y = "NANOG intensity (normalized)", title = title) +
  settheme +
  #scale_y_continuous(limits = c(0, NA), expand = c(0, 0)) +
  coord_flip() +
  scale_color_manual(values = palette) +
  scale_fill_manual(values = palette, guide = "none")  # avoid duplicate legend

ggsave(file.path(out_dir, sprintf("%s.pdf", title)), plot = p, width = w, height = h)
p


### G) GFP intensity per celltype (positive marker)

In [ ]:
#% of GFP+ cells in each lineage

summary_df <- merged_df %>%
  group_by(image, sample_name, condition, state) %>%
  summarise(
    c_GATA3     = sum(GATA3pos, na.rm = TRUE),
    c_NANOG     = sum(NANOGpos, na.rm = TRUE),
    c_neg     = sum(!GATA3pos & !NANOGpos, na.rm = TRUE),
    c_GATA3_GFP     = sum(GATA3pos & GFPpos, na.rm = TRUE),
    c_NANOG_GFP  = sum(NANOGpos & GFPpos, na.rm = TRUE),
    c_neg_GFP     = sum(!GATA3pos & !NANOGpos & GFPpos, na.rm = TRUE),
    .groups = "drop"
  )

summary_df$GATA3 = summary_df$c_GATA3_GFP / summary_df$c_GATA3
summary_df$NANOG = summary_df$c_NANOG_GFP / summary_df$c_NANOG
summary_df$double_neg = summary_df$c_neg_GFP / summary_df$c_neg


In [ ]:
summary_df

In [ ]:
head(summary_df)

In [ ]:
merged_long <- summary_df  %>%
  pivot_longer(
    cols = c(GATA3, NANOG, double_neg),
    names_to = "marker",
    values_to = "perGFPpos"
  )

In [ ]:
title = "G_perGFPpos_marker"
w <- 6
h <- 3
options(repr.plot.width=w, repr.plot.height=h)
  
p = ggplot(merged_long, aes(x = marker, y = perGFPpos, group = state)) +  # dots for each file
    stat_summary(aes(fill = state),
      fun = mean, 
      geom = "bar",
      position = position_dodge(width = 0.75),
      alpha = 0.4, width = 0.6) +   # error bars
    geom_jitter(
      aes(color = condition),
      position = position_jitterdodge(jitter.width = 0.1, dodge.width = 0.75),
      size = 1, alpha = 0.8
    )  +
    labs(
      title = title,
      y = "%GFP",
      x = ""
    )+ settheme+
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1)
    ) +
      scale_y_continuous(limits = c(0,1.1), expand = c(0, 0))+ 
      facet_grid(~condition)+
      scale_color_manual(values=col_condition)+
  scale_fill_manual(
    values = c(Developed = "grey30", Failed = "grey60"),
    name = "state"
  ) 

ggsave(file.path(out_dir, sprintf("%s.pdf", title)),
                plot = p, width = w, height = h)

p

In [ ]:
title = "G_perGFPpos_marker_sub"
w <- 3
h <- 2.5
options(repr.plot.width=w, repr.plot.height=h)

merged_long_sub = merged_long %>% subset(condition == "Grev_R")

p = ggplot(merged_long_sub, aes(x = marker, y = perGFPpos, group = state)) +  # dots for each file
    stat_summary(aes(fill = state),
      fun = mean, 
      geom = "bar",
      position = position_dodge(width = 0.75),
      alpha = 0.4, width = 0.6) +   # error bars
    geom_jitter(
      aes(color = condition),
      position = position_jitterdodge(jitter.width = 0.1, dodge.width = 0.75),
      size = 2, alpha = 0.8
    )  +
    labs(
      title = title,
      y = "%GFP",
      x = ""
    )+ settheme+
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1)
    ) +
      #scale_y_continuous(limits = c(0,0.4), expand = c(0, 0))+ 
      facet_grid(~condition)+
      scale_color_manual(values=col_condition)+
  scale_fill_manual(
    values = c(Developed = "grey30", Failed = "grey60"),
    name = "state"
  ) 

ggsave(file.path(out_dir, sprintf("%s.pdf", title)),
                plot = p, width = w, height = h)

p

### H) Intensity GFP+/-

In [ ]:
title   <- "H_GATA3norm_intensity_in_GATA3pos_GFP"
w <- 6; h <- 3
options(repr.plot.width=w, repr.plot.height=h)
palette <- col_condition

p <- ggplot(merged_df %>% filter(GATA3pos == TRUE), aes(x = forcats::fct_rev(condition), y = GATA3_norm)) +
  # points with jitter+dodge (no width/height here)
  geom_jitter(aes(color = GFPpos),
    size = 0.7, alpha = 0.6,
    position = position_jitterdodge(jitter.width = 0.2, dodge.width = 0.6),
    na.rm = TRUE
  ) +
  geom_boxplot(
    aes(fill = GFPpos),
    width = 0.5, alpha = 0.25, outlier.shape = NA,
    position = position_dodge(width = 0.6)
  ) +
  labs(x = "", y = "GATA3 intensity (normalized)", title = title) +
  settheme +
  facet_wrap(~state)+
  #scale_y_continuous(limits = c(0, NA), expand = c(0, 0)) +
  #coord_flip() 
  scale_color_manual(values = col_GFP) +
  scale_fill_manual(values = palette, guide = "none")  # avoid duplicate legend

ggsave(file.path(out_dir, sprintf("%s.pdf", title)), plot = p, width = w, height = h)
p

In [ ]:
title   <- "H_NANOG_norm_intensity_in_NANOGpos_GFP"
w <- 6; h <- 3
options(repr.plot.width=w, repr.plot.height=h)
palette <- col_condition

p <- ggplot(merged_df %>% filter(NANOGpos == TRUE), aes(x = forcats::fct_rev(condition), y = NANOG_norm)) +
  # points with jitter+dodge (no width/height here)
  geom_jitter(aes(color = GFPpos),
    size = 0.7, alpha = 0.6,
    position = position_jitterdodge(jitter.width = 0.2, dodge.width = 0.6),
    na.rm = TRUE
  ) +
  geom_boxplot(
    aes(fill = GFPpos),
    width = 0.5, alpha = 0.25, outlier.shape = NA,
    position = position_dodge(width = 0.6)
  ) +
  labs(x = "", y = "NANOG intensity (normalized)", title = title) +
  settheme +
  facet_wrap(~state)+
  #scale_y_continuous(limits = c(0, NA), expand = c(0, 0)) +
  #coord_flip() 
  scale_color_manual(values = col_GFP) +
  scale_fill_manual(values = palette, guide = "none")  # avoid duplicate legend

ggsave(file.path(out_dir, sprintf("%s.pdf", title)), plot = p, width = w, height = h)
p

### E) Normalise intensity by average of control

In [ ]:
head(merged_df)

In [ ]:
# Calculate the average values for the control
ctrl_vals <- merged_df %>%
  filter(condition_2 == "control") %>%
  summarise(
    mean_NANOG = mean(NANOG_norm, na.rm = TRUE),
    mean_GATA3 = mean(GATA3_norm, na.rm = TRUE)
  )

# Add these averages as columns and create the normalized columns
merged_df <- merged_df %>%
  mutate(
    ctrl_NANOG = ctrl_vals$mean_NANOG,
    ctrl_GATA3 = ctrl_vals$mean_GATA3,
    
    # Create the ratio columns
    NANOG_norm_ctr = NANOG_norm / ctrl_NANOG,
    GATA3_norm_ctr = GATA3_norm / ctrl_GATA3
  )

# Check the results
head(merged_df)

## Save

In [ ]:
head(merged_df)

In [ ]:
out_dir

In [ ]:
write.csv(merged_df, file.path(out_dir, sprintf("%s_%s_analysis_summary.csv", EXP, EXP_SUB)), row.names = FALSE)